In [1]:
!pip -q install -U ultralytics

import torch, ultralytics
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.114
Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [2]:
from pathlib import Path
import yaml, zipfile, shutil

def find_dataset_roots(base):
    roots = []
    for yml in Path(base).rglob("data.yaml"):
        root = yml.parent
        ok = all((root / s / "images").exists() and (root / s / "labels").exists()
                 for s in ["train", "valid", "test"])
        if ok:
            roots.append(root)
    return roots

roots = find_dataset_roots("/kaggle/input")

if not roots:
    zips = list(Path("/kaggle/input").rglob("*.zip"))
    assert zips, "Không tìm thấy data.yaml hoặc .zip trong /kaggle/input"
    zips = sorted(zips, key=lambda p: "yolov9" not in str(p).lower())
    extract_dir = Path("/kaggle/working/extracted_dataset")
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0], "r") as z:
        z.extractall(extract_dir)
    roots = find_dataset_roots(extract_dir)

preferred = [r for r in roots if "yolov9" in str(r).lower()]
DATA_ROOT = preferred[0] if preferred else roots[0]

with open(DATA_ROOT / "data.yaml", "r") as f:
    raw = yaml.safe_load(f)

names = raw["names"]
if isinstance(names, dict):
    names = [names[i] if i in names else names[str(i)] for i in range(len(names))]

DATA = {
    "path": str(DATA_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": int(raw.get("nc", len(names))),
    "names": names,
}

DATA_YAML = Path("/kaggle/working/garbage_yolov9_data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(DATA, f, sort_keys=False)

print("DATA_ROOT:", DATA_ROOT)
print(DATA_YAML.read_text())

DATA_ROOT: /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov9
path: /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION
  3.v2-gc1.yolov9
train: train/images
val: valid/images
test: test/images
nc: 6
names:
- BIODEGRADABLE
- CARDBOARD
- GLASS
- METAL
- PAPER
- PLASTIC



In [3]:
from collections import Counter
import pandas as pd

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
summary, dist = [], []

for split in ["train", "valid", "test"]:
    img_dir = DATA_ROOT / split / "images"
    lab_dir = DATA_ROOT / split / "labels"

    images = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    labels = list(lab_dir.glob("*.txt"))

    counter = Counter()
    for txt in labels:
        for line in txt.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1

    summary.append({"split": split, "images": len(images), "label_files": len(labels), "boxes": sum(counter.values())})

    for cid, cname in enumerate(names):
        dist.append({"split": split, "class_id": cid, "class_name": cname, "boxes": counter[cid]})

display(pd.DataFrame(summary))
display(pd.DataFrame(dist))

,split,images,label_files,boxes
0,train,7324,7324,51611
1,valid,2098,2098,18916
2,test,1042,1042,3563


,split,class_id,class_name,boxes
0,train,0,BIODEGRADABLE,31721
1,train,1,CARDBOARD,3386
2,train,2,GLASS,5429
3,train,3,METAL,3948
4,train,4,PAPER,2981
5,train,5,PLASTIC,4146
6,valid,0,BIODEGRADABLE,13637
7,valid,1,CARDBOARD,1292
8,valid,2,GLASS,2380
9,valid,3,METAL,1360


In [4]:
IMG = 640
BATCH = 16       # nếu OOM thì giảm 8 hoặc đổi MODEL = "yolov9t.pt"
EPOCHS = 50
MODEL = "yolov9s.pt"
RUN_NAME = "garbage_yolov9s"

model = YOLO(MODEL)

train_results = model.train(
    data=str(DATA_YAML),
    task="detect",
    imgsz=IMG,
    epochs=EPOCHS,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,
    project="/kaggle/working/runs_yolov9",
    name=RUN_NAME,
    patience=20,
    seed=42,
    plots=True,
    exist_ok=True,
)

Ultralytics 8.4.114 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/garbage_yolov9_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov9s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=garbage_yolov9s, nbs=

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

BEST = Path(f"/kaggle/working/runs_yolov9/{RUN_NAME}/weights/best.pt")
assert BEST.exists(), BEST

VAL_PROJECT = Path("/kaggle/working/runs_yolov9_val")
VAL_NAME = "garbage_test"

model = YOLO(str(BEST))

test_metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG,
    batch=BATCH,
    conf=0.001,
    iou=0.6,
    max_det=300,
    save_txt=True,
    save_conf=True,
    plots=True,
    project=str(VAL_PROJECT),
    name=VAL_NAME,
    exist_ok=True,
)

box = test_metrics.box

official_df = pd.DataFrame([{
    "Precision": float(box.mp),
    "Recall": float(box.mr),
    "F1_from_P_R": float(2 * box.mp * box.mr / (box.mp + box.mr + 1e-9)),
    "mAP@0.5": float(box.map50),
    "mAP@0.5:0.95": float(box.map),
}])

def as_1d(x):
    arr = np.asarray(x, dtype=float)
    if arr.ndim == 0:
        arr = np.array([float(arr)])
    return arr.reshape(-1)

def expand_metric(values, class_indices, nc, fill=np.nan):
    values = as_1d(values)
    class_indices = np.asarray(class_indices, dtype=int).reshape(-1)
    out = np.full(nc, fill, dtype=float)

    if len(values) == nc:
        out[:] = values[:nc]
    else:
        for i, cid in enumerate(class_indices[:len(values)]):
            if 0 <= cid < nc:
                out[cid] = values[i]
    return out

nc = len(names)
class_indices = getattr(box, "ap_class_index", np.arange(nc))

per_class_df = pd.DataFrame({
    "class_id": list(range(nc)),
    "class_name": names,
    "Precision": expand_metric(getattr(box, "p", []), class_indices, nc),
    "Recall": expand_metric(getattr(box, "r", []), class_indices, nc),
    "F1-score": expand_metric(getattr(box, "f1", []), class_indices, nc),
    "AP@0.5": expand_metric(getattr(box, "ap50", []), class_indices, nc),
    "AP@0.5:0.95": expand_metric(getattr(box, "ap", []), class_indices, nc),
})

display(official_df)
display(per_class_df)

official_df.to_csv("/kaggle/working/yolov9_official_test_summary.csv", index=False)
per_class_df.to_csv("/kaggle/working/yolov9_official_test_per_class.csv", index=False)

Ultralytics 8.4.114 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv9s summary (fused): 197 layers, 7,169,410 parameters, 0 gradients, 26.7 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.9 ms, read: 3.3±1.0 MB/s, size: 24.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov9/test/labels... 1042 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1042/1042 349.0it/s 3.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov9/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 66/66 1.6it/s 40.5s
                   all       1042       3563      0.575      0.433      0.464      0.345
    

,Precision,Recall,F1_from_P_R,mAP@0.5,mAP@0.5:0.95
0,0.57535,0.432507,0.493806,0.463997,0.34512


,class_id,class_name,Precision,Recall,F1-score,AP@0.5,AP@0.5:0.95
0,0,BIODEGRADABLE,0.275317,0.204082,0.234407,0.139322,0.056870
1,1,CARDBOARD,0.097986,0.300000,0.147723,0.136326,0.127076
2,2,GLASS,NaN,NaN,NaN,NaN,NaN
3,3,METAL,0.841370,0.636877,0.724979,0.691934,0.560517
4,4,PAPER,0.841498,0.474576,0.606888,0.637817,0.495412
5,5,PLASTIC,0.820580,0.547003,0.656428,0.714587,0.485725


In [6]:
import numpy as np
import pandas as pd

IOU_MATCH = 0.50
CONF_FOR_PRF1 = 0.25
IOU_THRESHOLDS = np.round(np.arange(0.50, 0.96, 0.05), 2)

GT_DIR = DATA_ROOT / "test" / "labels"
PRED_DIR = VAL_PROJECT / VAL_NAME / "labels"
assert PRED_DIR.exists(), PRED_DIR

def read_yolo_txt(path, pred=False):
    cols = 6 if pred else 5
    if not path.exists():
        return np.zeros((0, cols), dtype=float)
    rows = []
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if parts:
            rows.append([float(x) for x in parts[:cols]])
    if not rows:
        return np.zeros((0, cols), dtype=float)
    arr = np.array(rows, dtype=float)
    if pred and arr.shape[1] == 5:
        arr = np.c_[arr, np.ones(len(arr))]
    return arr

def xywh_to_xyxy(x):
    y = x.copy()
    y[:, 0] = x[:, 0] - x[:, 2] / 2
    y[:, 1] = x[:, 1] - x[:, 3] / 2
    y[:, 2] = x[:, 0] + x[:, 2] / 2
    y[:, 3] = x[:, 1] + x[:, 3] / 2
    return np.clip(y, 0, 1)

def box_iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[:, :, 0] * wh[:, :, 1]
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def compute_ap(recall, precision):
    if len(recall) == 0:
        return 0.0
    mrec = np.r_[0.0, recall, 1.0]
    mpre = np.r_[1.0, precision, 0.0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    x = np.linspace(0, 1, 101)
    return float(np.trapz(np.interp(x, mrec, mpre), x))

image_stems = sorted(
    {p.stem for p in (DATA_ROOT / "test" / "images").iterdir() if p.suffix.lower() in IMG_EXTS}
    | {p.stem for p in GT_DIR.glob("*.txt")}
    | {p.stem for p in PRED_DIR.glob("*.txt")}
)

GT, PRED = {}, {}

for stem in image_stems:
    g = read_yolo_txt(GT_DIR / f"{stem}.txt", pred=False)
    p = read_yolo_txt(PRED_DIR / f"{stem}.txt", pred=True)

    GT[stem] = (g[:, 0].astype(int), xywh_to_xyxy(g[:, 1:5])) if len(g) else (
        np.array([], dtype=int), np.zeros((0, 4))
    )
    PRED[stem] = (p[:, 0].astype(int), xywh_to_xyxy(p[:, 1:5]), p[:, 5]) if len(p) else (
        np.array([], dtype=int), np.zeros((0, 4)), np.array([])
    )

def collect_class(cid):
    gts, preds, n_gt = {}, [], 0
    for stem in image_stems:
        gc, gb = GT[stem]
        pc, pb, ps = PRED[stem]
        gboxes = gb[gc == cid]
        gts[stem] = gboxes
        n_gt += len(gboxes)
        for box, score in zip(pb[pc == cid], ps[pc == cid]):
            preds.append((stem, float(score), box))
    preds.sort(key=lambda x: x[1], reverse=True)
    return gts, preds, n_gt

def match_predictions(gts, preds, iou_thr, conf_min=None):
    if conf_min is not None:
        preds = [p for p in preds if p[1] >= conf_min]

    used = {stem: np.zeros(len(boxes), dtype=bool) for stem, boxes in gts.items()}
    tp = np.zeros(len(preds))
    fp = np.zeros(len(preds))
    matched_ious = []

    for i, (stem, score, box) in enumerate(preds):
        boxes = gts.get(stem, np.zeros((0, 4)))
        if len(boxes):
            ious = box_iou_matrix(box[None, :], boxes)[0]
            best_idx = int(np.argmax(ious))
            best_iou = float(ious[best_idx])
        else:
            best_idx, best_iou = -1, 0.0

        if best_iou >= iou_thr and best_idx >= 0 and not used[stem][best_idx]:
            tp[i] = 1
            used[stem][best_idx] = True
            matched_ious.append(best_iou)
        else:
            fp[i] = 1

    return tp, fp, matched_ious

def evaluate_at_iou(iou_thr):
    rows = []
    total_tp = total_fp = total_fn = 0
    all_ious, ap_values = [], []

    for cid, cname in enumerate(names):
        gts, preds, n_gt = collect_class(cid)

        tp_all, fp_all, _ = match_predictions(gts, preds, iou_thr)
        recall_curve = np.cumsum(tp_all) / (n_gt + 1e-9)
        precision_curve = np.cumsum(tp_all) / (np.cumsum(tp_all) + np.cumsum(fp_all) + 1e-9)
        ap = compute_ap(recall_curve, precision_curve) if n_gt > 0 else np.nan

        tp_fixed, fp_fixed, matched_ious = match_predictions(gts, preds, iou_thr, conf_min=CONF_FOR_PRF1)

        tp = int(tp_fixed.sum())
        fp = int(fp_fixed.sum())
        fn = int(n_gt - tp)

        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)

        rows.append({
            "class_id": cid,
            "class_name": cname,
            "gt_boxes": n_gt,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "mean_matched_IoU": float(np.mean(matched_ious)) if matched_ious else 0.0,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "AP": ap,
        })

        if n_gt > 0:
            ap_values.append(ap)
        total_tp += tp
        total_fp += fp
        total_fn += fn
        all_ious.extend(matched_ious)

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall = total_tp / (total_tp + total_fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    summary = {
        "IoU_threshold": iou_thr,
        "confidence_for_P_R_F1": CONF_FOR_PRF1,
        "mean_matched_IoU": float(np.mean(all_ious)) if all_ious else 0.0,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "mAP": float(np.nanmean(ap_values)),
    }

    return pd.DataFrame(rows), summary

per_class_50, summary_50 = evaluate_at_iou(IOU_MATCH)

map_curve = []
for t in IOU_THRESHOLDS:
    _, s = evaluate_at_iou(float(t))
    map_curve.append({"IoU_threshold": float(t), "mAP": s["mAP"]})

map_curve = pd.DataFrame(map_curve)
summary_50["mAP@0.5:0.95_custom"] = map_curve["mAP"].mean()

display(per_class_50)
display(pd.DataFrame([summary_50]))
display(map_curve)

per_class_50.to_csv("/kaggle/working/yolov9_test_per_class_metrics_iou50.csv", index=False)
pd.DataFrame([summary_50]).to_csv("/kaggle/working/yolov9_test_summary_metrics.csv", index=False)
map_curve.to_csv("/kaggle/working/yolov9_test_map_by_iou.csv", index=False)

/tmp/ipykernel_23/307083034.py:54: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(np.interp(x, mrec, mpre), x))


,class_id,class_name,gt_boxes,tp,fp,fn,mean_matched_IoU,Precision,Recall,F1-score,AP
0,0,BIODEGRADABLE,49,14,78,35,0.729490,0.152174,0.285714,0.198582,0.140085
1,1,CARDBOARD,20,8,105,12,0.923332,0.070796,0.400000,0.120301,0.136790
2,2,GLASS,0,0,89,0,0.000000,0.000000,0.000000,0.000000,NaN
3,3,METAL,533,352,106,181,0.901814,0.768559,0.660413,0.710394,0.705673
4,4,PAPER,1376,730,230,646,0.903870,0.760417,0.530523,0.625000,0.646113
5,5,PLASTIC,1585,986,381,599,0.853230,0.721287,0.622082,0.668022,0.719025


,IoU_threshold,confidence_for_P_R_F1,mean_matched_IoU,Precision,Recall,F1-score,mAP,mAP@0.5:0.95_custom
0,0.5,0.25,0.87854,0.678792,0.586584,0.629329,0.469537,0.35317


,IoU_threshold,mAP
0,0.50,0.469537
1,0.55,0.456781
2,0.60,0.441139
3,0.65,0.418241
4,0.70,0.393284
5,0.75,0.359545
6,0.80,0.325632
7,0.85,0.285538
8,0.90,0.237647
9,0.95,0.144351
